In [ ]:
import pandas as pd

df_tayara = pd.read_csv("../data/raw/tayara.csv")
df_mubawab = pd.read_csv("../data/raw/mubawab.csv")
df_immobilier = pd.read_csv("../data/raw/immobilier.csv")

# 1.Data Preprocessing


In [ ]:

cities_mubawab = df_mubawab.iloc[:, -1].str.strip()
unique_cities_mubawab = sorted(cities_mubawab.unique())

cities_tayara = df_tayara.iloc[:, -1].str.strip()
unique_cities_tayara = sorted(cities_tayara.unique())

cities_immobilier = df_immobilier.iloc[:, -1].str.strip()
unique_cities_immobilier = sorted(
    df_immobilier.iloc[:, -1]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print(unique_cities_mubawab)
print(unique_cities_tayara)
print(unique_cities_immobilier)


### Data Inspection


In [ ]:
df_mubawab.info()
df_mubawab.head()

df_tayara.info()
df_tayara.head()

df_immobilier.info()
df_immobilier.head()


###  Merge

In [ ]:
import pandas as pd

df_all = pd.concat(
    [df_mubawab, df_tayara, df_immobilier],
    ignore_index=True
)

print(df_all.shape)


### Normalize City Names

In [ ]:
import re
import unidecode

def normalize_city(city):
    if not isinstance(city, str):
        return None
    city = city.strip().lower()
    city = unidecode.unidecode(city)
    city = re.sub(r"\s+", " ", city)
    return city

df_all["city_raw"] = df_all["ville"]
df_all["city_norm"] = df_all["city_raw"].apply(normalize_city)


### City to Government mapping

In [ ]:
CITY_TO_GOV = {
    # TUNIS
    "tunis": "Tunis",
    "la marsa": "Tunis",
    "carthage": "Tunis",
    "le bardo": "Tunis",
    "le kram": "Tunis",
    "la goulette": "Tunis",
    "el omrane": "Tunis",
    "el omrane superieur": "Tunis",
    "el kabaria": "Tunis",
    "el hrairia": "Tunis",
    "sidi hassine": "Tunis",

    # ARIANA
    "ariana": "Ariana",
    "raoued": "Ariana",
    "sidi thabet": "Ariana",
    "el menzah": "Ariana",
    "ennasr": "Ariana",

    # BEN AROUS
    "ben arous": "Ben Arous",
    "benarous": "Ben Arous",
    "rades": "Ben Arous",
    "ezzahra": "Ben Arous",
    "boumhel bassatine": "Ben Arous",
    "hammam chatt": "Ben Arous",
    "hammam lif": "Ben Arous",

    # MANOUBA
    "la manouba": "Manouba",
    "lamanouba": "Manouba",
    "douar hicher": "Manouba",

    # NABEUL
    "nabeul": "Nabeul",
    "hammamet": "Nabeul",
    "hammamet nord": "Nabeul",
    "korba": "Nabeul",
    "kelibia": "Nabeul",
    "takelsa": "Nabeul",
    "soliman": "Nabeul",
    "dar chaabane": "Nabeul",

    # SOUSSE
    "sousse": "Sousse",
    "sahloul": "Sousse",
    "khezama est": "Sousse",
    "khezama ouest": "Sousse",
    "jaouhara": "Sousse",
    "tantana": "Sousse",
    "hammam sousse": "Sousse",

    # SFAX
    "sfax": "Sfax",
    "sfax sud": "Sfax",
    "sfax ouest": "Sfax",

    # MONASTIR
    "monastir": "Monastir",
    "jemmal": "Monastir",
    "ksar hellal": "Monastir",

    # MAHDIA
    "mahdia": "Mahdia",
    "chebba": "Mahdia",

    # GABES
    "gabes": "Gabes",
    "gabs": "Gabes",

    # MEDNINE
    "medenine": "Mednine",
    "mdenine": "Mednine",
    "djerba": "Mednine",
    "houmt souk": "Mednine",
    "midoun": "Mednine",

    # KAIROUAN
    "kairouan": "Kairouan",

    # KASSERINE
    "kasserine": "Kasserine",

    # GAFSA
    "gafsa": "Gafsa",

    # TOZEUR
    "tozeur": "Tozeur",

    # ZAGHOUAN
    "zaghouan": "Zaghouan",
}


In [ ]:
df_all["governorate"] = df_all["city_norm"].map(CITY_TO_GOV)


### Drop ligns with empty column values

In [ ]:
before = df_all.shape[0]
df_all = df_all.dropna(subset=["governorate", "surface", "prix", "chambres"])
after = df_all.shape[0]

print(f"Dropped {before - after} rows")


In [ ]:
df_all["governorate"].value_counts()


### Dropping columns we don't need


In [ ]:
#drop type_transaction column as they are all a vendre
df_all = df_all.drop(columns=["type_transaction"])


In [ ]:
#drop city_raw column
df_all = df_all.drop(columns=["city_raw"])

In [ ]:
# drop source column
df_all = df_all.drop(columns=["source"])


### Renaming and saving final csv

In [ ]:

df_clean = df_all.rename(columns={
    "prix": "price",
    "surface": "surface",
    "chambres": "rooms",
    "type_bien": "property_type",
})

df_clean = df_clean[
    [
        "price",
        "surface",
        "rooms",
        "governorate",
        "property_type",
    ]
]
df_clean.info()
df_clean.to_csv("../data/clean/clean_housing_tunisia.csv", index=False)



# 2.EDA


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/clean/clean_housing_tunisia.csv")
df.head()
df.info()
df.describe()

### Description of the target (price)


In [ ]:
plt.figure(figsize=(6,4))
df["price"].hist(bins=50)
plt.title("Price distribution (raw)")
plt.xlabel("Price")
plt.ylabel("Count")
plt.show()


### Log transform price

In [ ]:
df["log_price"] = np.log1p(df["price"])

plt.figure(figsize=(6,4))
df["log_price"].hist(bins=50)
plt.title("Log(price) distribution")
plt.xlabel("log(price)")
plt.ylabel("Count")
plt.show()


### Surface and room inspection

#### Surface

In [ ]:
df["surface"].describe(percentiles=[0.5, 0.9, 0.95, 0.99])


In [ ]:
df["log_surface"] = np.log1p(df["surface"])

df["log_surface"].hist(bins=50)
plt.title("Log(surface) distribution")
plt.show()

#### Rooms

In [ ]:
df["rooms"].value_counts().sort_index()


### Categorical Variables

#### Governorate

In [ ]:
plt.figure(figsize=(12,4))
df["governorate"].value_counts().plot(kind="bar")
plt.title("Number of listings per governorate")
plt.ylabel("Count")
plt.show()

#### Property type

In [ ]:
df["property_type"].value_counts().plot(kind="bar")
plt.title("Property type distribution")
plt.show()


### Relationships

#### Price vs Surface (raw)

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(df["surface"], df["price"], alpha=0.3)
plt.xlabel("Surface")
plt.ylabel("Price")
plt.title("Price vs Surface")
plt.show()


#### Log to log relationship

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(df["log_surface"], df["log_price"], alpha=0.3)
plt.xlabel("Log(surface)")
plt.ylabel("Log(price)")
plt.title("Log(price) vs Log(surface)")
plt.show()


#### Price by government

In [ ]:
plt.figure(figsize=(14,5))
sns.boxplot(x="governorate", y="log_price", data=df)
plt.xticks(rotation=45)
plt.title("Log(price) by governorate")
plt.show()


# Cleaning unreasonable/impossible data

In [ ]:
df = df[(df["price"] > 10000) & (df["price"] < 1e7)]

df = df[(df["surface"] > 10) & (df["surface"] < 5000)]
df = df[(df["rooms"] >= 1) & (df["rooms"] <= 10)]
df["log_price"] = np.log1p(df["price"])
df["log_surface"] = np.log1p(df["surface"])
df[["price", "surface"]].describe(percentiles=[0.95, 0.99])


In [ ]:
df.to_csv("../data/clean/clean_housing_tunisia_model_ready.csv", index=False)


# 3.Modeling

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/clean/clean_housing_tunisia_model_ready.csv")

df.head()


### Define Target and features

In [ ]:
df["log_price"] = np.log1p(df["price"])

X = df[["surface", "rooms", "governorate", "property_type"]]
y = df["log_price"]


### Encode Categorical Values

In [ ]:
X = pd.get_dummies(
    X,
    columns=["governorate", "property_type"],
    drop_first=True
)


### Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Linear Regression RMSE:", rmse)
print("Linear Regression R²:", r2)


## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest RMSE:", rmse_rf)
print("Random Forest R²:", r2_rf)


### Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "RMSE": [rmse, rmse_rf],
    "R2": [r2, r2_rf]
})

results


#### Random forest hyperparameter tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [10, 20, 30, None],
    'min_samples_leaf': [1, 2, 4],
    'n_jobs': [-1]
}

# Instantiate Random Forest model
rf = RandomForestRegressor(random_state=42)

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=2)

# Fit grid search
grid_search.fit(X_train, y_train)

# Best hyperparameters
print("Best hyperparameters:", grid_search.best_params_)

# Best score (RMSE)
print("Best RMSE:", np.sqrt(-grid_search.best_score_))


### Changing random forest hyperparameters


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    random_state=42,
    min_samples_leaf=4,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest RMSE:", rmse_rf)
print("Random Forest R²:", r2_rf)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

importances = rf.feature_importances_

# Create a DataFrame for easy plotting
feat_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x="Importance", y="Feature", data=feat_importance)
plt.title("Feature Importance (Random Forest)")
plt.show()


## XGBoost

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    eval_metric='rmse',
    n_estimators=500,          
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42
)

# Train
xgb_model.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb_model.predict(X_test)

# Metrics
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print("XGBoost RMSE:", rmse_xgb)
print("XGBoost R²:", r2_xgb)


# 4.Tracking

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Load your data
df = pd.read_csv("../data/clean/clean_housing_tunisia_model_ready.csv")

# Feature engineering (optional, can toggle per run)
df["rooms_per_surface"] = df["rooms"] / df["surface"]
df["surface_squared"] = df["surface"] ** 2

# Pick features
X = df[
    ["surface", "surface_squared", "rooms", "rooms_per_surface",
     "governorate", "property_type"]
]
X = pd.get_dummies(X, columns=["governorate", "property_type"], drop_first=True)
y = df["log_price"]

# Split function (toggle between random or grouped)
def get_split(X, y, method="random"):
    if method == "grouped":
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        for train_idx, test_idx in gss.split(X, y, groups=df["governorate"]):
            return X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]
    else:
        return train_test_split(X, y, test_size=0.2, random_state=42)

# Models to test
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(
        n_estimators=300, max_depth=None, min_samples_leaf=1, random_state=42, n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        objective="reg:squarederror", random_state=42, n_jobs=-1
    )
}
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("house-price-tunisia")

# Iterate over models and split types
for split_method in ["random", "grouped"]:
    X_train, X_test, y_train, y_test = get_split(X, y, method=split_method)
    
    for name, model in models.items():
        run_name = f"{name}_{split_method}_split"
        with mlflow.start_run(run_name=run_name):
            # Train
            model.fit(X_train, y_train)
            
            # Predict
            y_pred = model.predict(X_test)
            
            # Metrics
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            r2 = r2_score(y_test, y_pred)
            
            # Log metrics and params
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_param("model", name)
            mlflow.log_param("split", split_method)
            mlflow.log_param("features", list(X.columns))
            
            # Log model
            
            mlflow.sklearn.log_model(model, name="model", skops_trusted_types=["sklearn.tree._tree.Tree", "xgboost.core.Booster", "xgboost.sklearn.XGBRegressor"])

# 5.Model_registry

### Import mlflow client

In [ ]:
from mlflow.tracking import MlflowClient
import pickle

client = MlflowClient()
mlflow.set_tracking_uri("sqlite:///mlflow.db")
client = mlflow.tracking.MlflowClient()


### Take the best run

In [ ]:
runs = client.search_runs(experiment_ids=["0"])  # or your experiment's actual ID
for r in runs:
    print(r.info.run_id, r.info.status)

In [ ]:
import os; print(os.path.exists('mlflow.db'), os.getcwd())

In [ ]:
client.search_experiments()

In [ ]:
runs = client.search_runs(experiment_ids=["1"])
for r in runs:
    print(r.info.run_id, r.info.status, r.data.metrics)

In [ ]:
# Choose best run
best_run_id = "ece9af47030f477f86d852e3e238f202"
best_run = client.get_run(best_run_id)

# Extract feature columns logged in this run
feature_columns = best_run.data.params["features"]
print("Feature columns of best model:", feature_columns)
import os
os.makedirs("artifacts", exist_ok=True)

with open("artifacts/best_model_features.pkl", "wb") as f:
    pickle.dump(feature_columns, f)
# Save feature columns for FastAPI / registry
with open("artifacts/best_model_features.pkl", "wb") as f:
    pickle.dump(feature_columns, f)

# Log them as artifact in MLflow
client.log_artifact(best_run_id, "artifacts/best_model_features.pkl", artifact_path="model_artifacts")

### Register the best model

In [ ]:


# Register model in the Model Registry
model_name = "HousePriceModel"

# Create the registered model 
try:
    client.create_registered_model(model_name)
except:
    print("Model already exists, skipping creation.")

# Create a new version for this run
version = client.create_model_version(
    name=model_name,
    source=f"runs:/{best_run_id}/model",
    run_id=best_run_id
)

print(f"Registered model version: {version.version}")


### Promote the version to production 

In [ ]:
client.transition_model_version_stage(
    name=model_name,
    version=version.version,
    stage="Production"
)

print(f"Model version {version.version} is now in Production")


